<a href="https://colab.research.google.com/github/LukeXboy/E-Commerce-Data-Analysis-Personal-Project/blob/main/Olist_Ecommerce_Revenue_Customer_Analytics_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E-Commerce Revenue & Customer Behavior Analysis
**Brazilian Olist marketplace | Independent portfolio case study | October 2026**

**Business question:** What drove the October-to-November 2017 revenue increase, and how did customer acquisition, product categories, geography, and purchase behavior contribute?

**Portfolio status:** Analysis completed at an **exploratory/descriptive scope**. The notebook is a **clean reproduction of previously conducted analyses**, prepared from documented calculations; it has **not been rerun against original raw files in this workspace**. The source dataset is required to reproduce results. No retention outcome, forecasting model, dashboard, inferential test, or causal effect is claimed.

Data: [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) (public research dataset; transactions mainly 2016–2018).

## 0. Setup and input files

Download the Olist CSV files from Kaggle (subject to the dataset license) and place them under `data/raw/`, relative to your project root. Launch Jupyter from the repository root. Do **not** commit downloaded customer/order records. This notebook requires Python 3.10+, `pandas`, `numpy`, and `duckdb` for the optional SQL cell.

Required filenames: `olist_orders_dataset.csv`, `olist_order_payments_dataset.csv`, `olist_customers_dataset.csv`, `olist_order_items_dataset.csv`, `olist_order_reviews_dataset.csv`, `olist_products_dataset.csv`, and `product_category_name_translation.csv`.

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path('.')
paths = {
    'orders': 'olist_orders_dataset.csv',
    'payments': 'olist_order_payments_dataset.csv',
    'customers': 'olist_customers_dataset.csv',
    'items': 'olist_order_items_dataset.csv',
    'reviews': 'olist_order_reviews_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'translations': 'product_category_name_translation.csv',
}
missing = [name for name in paths.values() if not (DATA_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing files: {missing}.')
tables = {key: pd.read_csv(DATA_DIR / filename) for key, filename in paths.items()}
orders, payments, customers = (tables[x] for x in ('orders', 'payments', 'customers'))
items, reviews, products, translations = (tables[x] for x in ('items','reviews','products','translations'))
for col in ('order_purchase_timestamp','order_approved_at','order_delivered_carrier_date',
            'order_delivered_customer_date','order_estimated_delivery_date'):
    orders[col] = pd.to_datetime(orders[col], errors='coerce')
print({name: frame.shape for name, frame in tables.items()})

{'orders': (99441, 8), 'payments': (103886, 5), 'customers': (99441, 5), 'items': (112650, 7), 'reviews': (99224, 7), 'products': (32951, 9), 'translations': (71, 2)}


## 1. Data quality and key integrity

The analytical grain is **one row per order**. Keep `customer_id` (order-scoped key) distinct from `customer_unique_id` (stable buyer identity); joining on the latter before aggregation can cause duplicate orders.

Inspect purchase and delivery timestamps separately. Delivered-status records may occasionally lack a delivery timestamp; they remain in the **delivered-status** KPI population but need an explicit exception for any actual delivery-time analysis.

In [4]:
assert orders['order_id'].is_unique, 'orders should contain exactly one row per order_id'
assert customers['customer_id'].is_unique, 'customers should have unique customer_id'
print('orders:', len(orders), 'unique IDs:', orders.order_id.nunique())
print('status counts:\n', orders.order_status.value_counts())
print('missing important timestamps:\n', orders[['order_approved_at',
    'order_delivered_carrier_date','order_delivered_customer_date']].isna().sum())
print('delivered orders with missing actual delivery timestamps:',
    orders.loc[orders.order_status.eq('delivered'), 'order_delivered_customer_date'].isna().sum())
print('duplicate payment order IDs (expected possible):', payments.order_id.duplicated().sum())
print('duplicate review order IDs (expected possible):', reviews.order_id.duplicated().sum())

orders: 99441 unique IDs: 99441
status counts:
 order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64
missing important timestamps:
 order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64
delivered orders with missing actual delivery timestamps: 8
duplicate payment order IDs (expected possible): 4446
duplicate review order IDs (expected possible): 551


## 2. Construct the one-row-per-order analytical dataset

Payment records and reviews are one-to-many relative to orders, so aggregate **before** joining. For revenue, sum the payment value per order rather than multiplying it by the number of items. The original analysis documented exactly **one delivered order with missing payment information**, `bfbd0f9bdef84302105ad712db648a6c`, where 3 observed line items at BRL 44.99 plus BRL 2.83 freight each produce a **reconstructed** BRL 143.46. The imputed value is not observed payment revenue and must be flagged. Do not silently fill other missing amounts.

In [5]:
payments_by_order = payments.groupby('order_id', as_index=False).agg(
    order_revenue=('payment_value','sum'), payment_records=('payment_value','size')
)
reviews_by_order = reviews.groupby('order_id', as_index=False).agg(
    avg_review_score=('review_score','mean'), review_records=('review_score','size')
)
base = orders.merge(customers[['customer_id','customer_unique_id','customer_state']],
                    on='customer_id', how='left', validate='many_to_one')
base = base.merge(payments_by_order, on='order_id', how='left', validate='one_to_one')
base = base.merge(reviews_by_order, on='order_id', how='left', validate='one_to_one')
assert len(base) == len(orders) and base.order_id.is_unique
assert base.customer_unique_id.notna().all()

base['revenue_reconstructed'] = False
exception_order = 'bfbd0f9bdef84302105ad712db648a6c'
if exception_order in set(base['order_id']):
    mask = base.order_id.eq(exception_order) & base.order_revenue.isna()
    if mask.any():
        item_subset = items.loc[items.order_id.eq(exception_order),['price','freight_value']]
        reconstructed = item_subset[['price','freight_value']].sum().sum()
        if not np.isclose(reconstructed, 143.46):
            raise ValueError(f'Exception reconstruction mismatch: {reconstructed}')
        base.loc[mask,'order_revenue'] = reconstructed
        base.loc[mask,'revenue_reconstructed'] = True
print('one row per order:', len(base), 'unique IDs:', base.order_id.nunique())
print('reconstructed payments:', int(base.revenue_reconstructed.sum()))
print('missing review averages (remain null):', int(base.avg_review_score.isna().sum()))
print('delivered missing order revenue:', int(base.loc[base.order_status.eq('delivered'),
                                                  'order_revenue'].isna().sum()))

one row per order: 99441 unique IDs: 99441
reconstructed payments: 1
missing review averages (remain null): 768
delivered missing order revenue: 0


## 3. Monthly KPI decomposition (delivered-status orders)

Use full year-month (`Period('M')`), not numeric month alone. **AOV = revenue / distinct orders**. Revenue may be decomposed descriptively as **unique customers × orders per customer × AOV**. These are associations in historical data, not causal attributions. Exclude partial boundary months from year-over-year growth conclusions.

In [6]:
delivered = base.loc[base.order_status.eq('delivered')].copy()
delivered['month'] = delivered.order_purchase_timestamp.dt.to_period('M')
monthly = delivered.groupby('month', as_index=False).agg(
    revenue=('order_revenue','sum'),
    orders=('order_id','nunique'),
    unique_customers=('customer_unique_id','nunique'),
    mean_review_score=('avg_review_score','mean')
)
monthly['aov'] = monthly.revenue / monthly.orders
monthly['orders_per_customer'] = monthly.orders / monthly.unique_customers
display(monthly.loc[monthly.month.isin([pd.Period('2017-10'),pd.Period('2017-11')])])

,month,revenue,orders,unique_customers,mean_review_score,aov,orders_per_customer
12,2017-10,751140.27,4478,4417,4.205241,167.740123,1.013810
13,2017-11,1153528.05,7289,7183,3.988878,158.256009,1.014757


In [7]:
comparison = monthly.set_index('month').loc[[pd.Period('2017-10'),pd.Period('2017-11')]]
october, november = comparison.iloc[0], comparison.iloc[1]
for metric in ('revenue','orders','unique_customers','orders_per_customer','aov'):
    pct = (november[metric] / october[metric] - 1)*100
    print(f'{metric:22s}: {pct:+.2f}%')
# Previously recorded: October R$751,140.27, November R$1,153,528.05;
# revenue +53.6%; unique customers +62.6%; AOV -5.7% approximately.
# Review discrepancies before changing the published narrative.

revenue               : +53.57%
orders                : +62.77%
unique_customers      : +62.62%
orders_per_customer   : +0.09%
aov                   : -5.65%


### SQL cross-check (optional DuckDB)

SQL is run against the same delivered order-grain dataframe; **do not join raw payments or raw order items before calculating revenue**.

In [9]:
try:
    import duckdb
except ImportError:
    print('Optional DuckDB step skipped (pip install duckdb).')
else:
    # Drop the custom Period column 'month' before registering to avoid conversion errors in DuckDB
    duckdb.register('delivered_orders', delivered.drop(columns=['month']))
    sql_check = duckdb.sql("""
        SELECT CAST(DATE_TRUNC('month', order_purchase_timestamp) AS DATE) AS month,
               COUNT(DISTINCT order_id) AS orders,
               COUNT(DISTINCT customer_unique_id) AS customers,
               ROUND(SUM(order_revenue),2) AS revenue,
               ROUND(SUM(order_revenue)/NULLIF(COUNT(DISTINCT order_id),0),2) AS aov
        FROM delivered_orders
        WHERE order_purchase_timestamp >= '2017-10-01'
          AND order_purchase_timestamp < '2017-12-01'
        GROUP BY 1 ORDER BY 1
    """)
    display(sql_check.df())

,month,orders,customers,revenue,aov
0,2017-10-01,4478,4417,751140.27,167.74
1,2017-11-01,7289,7183,1153528.05,158.26


## 4. Category and state growth contributions

**Category `item_revenue` means sum of order-item selling prices, not total payment revenue.** Payment records may include freight, vouchers, and multiple payment methods. Category-level changes should therefore not be summed and described as equivalent to order-payment revenue growth.

In [10]:
month_orders = delivered.loc[delivered.month.isin([pd.Period('2017-10'),pd.Period('2017-11')]),
    ['order_id','month','customer_state']].copy()
month_items = month_orders.merge(items[['order_id','order_item_id','product_id','price','freight_value']],
                                on='order_id', how='left', validate='one_to_many')
month_items = month_items.merge(products[['product_id','product_category_name']],
                                on='product_id', how='left', validate='many_to_one')
month_items = month_items.merge(translations[['product_category_name','product_category_name_english']],
                                on='product_category_name', how='left', validate='many_to_one')
month_items['category'] = month_items.product_category_name_english.fillna('Unknown / untranslated')
category_growth = month_items.groupby(['category','month'],as_index=False).agg(
    item_revenue=('price','sum'), items_sold=('order_item_id','count'),
    orders=('order_id','nunique')
).pivot(index='category', columns='month', values='item_revenue').fillna(0)
category_growth['delta'] = category_growth.get(pd.Period('2017-11'),0) - category_growth.get(pd.Period('2017-10'),0)
display(category_growth.sort_values('delta',ascending=False).head(10))

state_growth = delivered.loc[delivered.month.isin([pd.Period('2017-10'),pd.Period('2017-11')])].groupby(
    ['customer_state','month'],as_index=False).agg(revenue=('order_revenue','sum'))
state_growth = state_growth.pivot(index='customer_state',columns='month',values='revenue').fillna(0)
state_growth['delta'] = state_growth.get(pd.Period('2017-11'),0) - state_growth.get(pd.Period('2017-10'),0)
display(state_growth.sort_values('delta',ascending=False).head(10))

month,2017-10,2017-11,delta
category,,,
bed_bath_table,46007.70,87957.63,41949.93
health_beauty,40698.99,78274.40,37575.41
furniture_decor,30009.74,62091.27,32081.53
watches_gifts,64874.63,95292.34,30417.71
toys,33324.42,62611.26,29286.84
computers_accessories,42009.38,69676.32,27666.94
garden_tools,25798.75,44275.18,18476.43
housewares,15883.68,33872.42,17988.74
perfumery,16877.82,31612.13,14734.31


month,2017-10,2017-11,delta
customer_state,,,
SP,249924.09,401027.72,151103.63
RJ,110577.92,172234.57,61656.65
MG,93049.17,154131.98,61082.81
RS,43173.06,67262.74,24089.68
SC,24773.77,44269.59,19495.82
PR,37404.60,54237.96,16833.36
ES,12667.55,28402.83,15735.28
DF,14731.50,30196.02,15464.52
BA,27614.20,35449.55,7835.35


## 5. New versus returning customers in four states

Focus on **SP, RJ, MG, RS**. Define first-observed customer using the earliest *delivered-status purchase* **across the whole observed history and all states**. A buyer with no previous delivered order in the dataset is *new within observed delivered history*, not necessarily new to the marketplace in reality.

In [11]:
first_delivered = delivered.groupby('customer_unique_id')['order_purchase_timestamp'].min()
selected_states = ['SP','RJ','MG','RS']

def monthly_new_cohort(period):
    start = period.to_timestamp()
    end = (period+1).to_timestamp()
    cohort_orders = delivered.loc[delivered.month.eq(period) &
        delivered.customer_state.isin(selected_states)].copy()
    prior_first = cohort_orders.customer_unique_id.map(first_delivered)
    cohort_orders['customer_label'] = np.where(prior_first >= start,'new','returning')
    new_orders = cohort_orders.loc[cohort_orders.customer_label.eq('new')].copy()
    assert new_orders.order_id.is_unique
    new_ids = new_orders.customer_unique_id.nunique()
    return cohort_orders, new_orders, new_ids

nov_four_states, nov_new_orders, nov_new_count = monthly_new_cohort(pd.Period('2017-11'))
oct_four_states, oct_new_orders, oct_new_count = monthly_new_cohort(pd.Period('2017-10'))
print('all November four-state orders:',len(nov_four_states),
      'new customers:',nov_new_count, 'new-customer orders:',len(nov_new_orders))
display(nov_four_states.groupby(['customer_state','customer_label'])['customer_unique_id'].nunique())
# Previously recorded: Nov four-state orders 5,242; new customers 5,074.

all November four-state orders: 5242 new customers: 5074 new-customer orders: 5153


customer_state  customer_label
MG              new                893
                returning           21
RJ              new                979
                returning           17
RS              new                399
                returning            6
SP              new               2803
                returning           41
Name: customer_unique_id, dtype: int64

## 6. Category reach among **new customers**

Categories overlap at the customer level. For top-five reach, take the **union** of purchasers and then count unique customer IDs; do not sum category-level customer counts. The top-five categories are selected by **November's new customer reach** and held fixed for October comparison.

In [12]:
def enrich_order_items(order_frame):
    x = order_frame[['order_id','customer_unique_id']].merge(
        items[['order_id','order_item_id','product_id','price']],
        on='order_id',how='left',validate='one_to_many')
    x = x.merge(products[['product_id','product_category_name']],
                on='product_id',how='left',validate='many_to_one')
    x = x.merge(translations[['product_category_name','product_category_name_english']],
                on='product_category_name',how='left',validate='many_to_one')
    return x

nov_new_items = enrich_order_items(nov_new_orders)
nov_category = (nov_new_items.groupby('product_category_name_english',as_index=False)
   .agg(unique_new_customers=('customer_unique_id','nunique'),
        orders=('order_id','nunique'),items_sold=('order_item_id','count'),
        item_revenue=('price','sum'))
   .sort_values('unique_new_customers',ascending=False))
top5 = nov_category.head(5).product_category_name_english.tolist()
print('top-five November new customer categories:',top5)
display(nov_category.head(10))

def top5_reach(new_orders, top5_categories):
    new_items = enrich_order_items(new_orders)
    purchasers = new_items.loc[new_items.product_category_name_english.isin(top5_categories),
                               'customer_unique_id'].nunique()
    denominator = new_orders.customer_unique_id.nunique()
    return dict(new_customers=denominator, top5_customers=purchasers,
                reach_pct=100 * purchasers/denominator)

reach_oct = top5_reach(oct_new_orders, top5)
reach_nov = top5_reach(nov_new_orders, top5)
display(pd.DataFrame([reach_oct,reach_nov],index=['2017-10','2017-11']))
# Previously observed: Oct 1176/3052 = 38.53%; Nov 2025/5074 = 39.91%.
# Note: If category translations are unavailable, those categories remain missing.
print('Nov item rows with missing English categories:',
      nov_new_items.product_category_name_english.isna().sum())

top-five November new customer categories: ['bed_bath_table', 'furniture_decor', 'sports_leisure', 'health_beauty', 'toys']


,product_category_name_english,unique_new_customers,orders,items_sold,item_revenue
6,bed_bath_table,610,621,735,68343.26
36,furniture_decor,388,395,555,43879.82
58,sports_leisure,366,376,412,44080.30
40,health_beauty,336,340,373,51087.67
62,toys,335,336,375,46704.16
39,garden_tools,303,306,377,31050.56
14,computers_accessories,286,289,332,43943.76
63,watches_gifts,279,281,310,62330.03
46,housewares,259,260,316,26309.70
61,telephony,211,211,225,15222.68


,new_customers,top5_customers,reach_pct
2017-10,3052,1176,38.532110
2017-11,5074,2025,39.909342


Nov item rows with missing English categories: 84


## 7. Customer-group AOV comparison

Create two mutually exclusive customer groups: **Top5 buyer** (any purchase in the fixed top five) and **Other buyer**. Assign a customer once, then join back to the **order** dataframe, which guarantees no item-level multiplication of total order revenue. AOV is *revenue / orders*, not revenue / customers.

In [13]:
top5_ids = set(nov_new_items.loc[
    nov_new_items.product_category_name_english.isin(top5),'customer_unique_id'])
grouped_customers = nov_new_orders[['customer_unique_id']].drop_duplicates().copy()
grouped_customers['customer_group'] = np.where(
    grouped_customers.customer_unique_id.isin(top5_ids),'Top5 buyer','Other buyer')
segmented = nov_new_orders.merge(grouped_customers,on='customer_unique_id',
                                 how='left',validate='many_to_one')
assert len(segmented) == len(nov_new_orders)
assert segmented.order_id.is_unique
assert segmented.customer_group.notna().all()
summary = segmented.groupby('customer_group',as_index=False).agg(
    unique_customers=('customer_unique_id','nunique'),
    orders=('order_id','nunique'),
    total_revenue=('order_revenue','sum'))
summary['aov'] = summary.total_revenue / summary.orders
display(summary)
# Previously observed: Other buyer: 3,049 customers, 3,073 orders,
# R$484,474.40 revenue, R$157.66 AOV.
# Top5 buyer: 2,025 customers, 2,080 orders,
# R$299,439.32 revenue, R$143.96 AOV.

,customer_group,unique_customers,orders,total_revenue,aov
0,Other buyer,3049,3073,484474.40,157.655190
1,Top5 buyer,2025,2080,299439.32,143.961212


## 8. Findings and limitations

**Previously observed (original analyst notebook and exercises; must be reproduced from raw data):**

- October → November 2017 delivered-order revenue: **BRL 751,140.27 → 1,153,528.05 (+53.6%)**.
- Unique customers grew approximately **62.6%** while AOV fell approximately **5.7%**. Revenue growth was primarily **volume-driven**, not higher per-order spending.
- Largest increases in state-level delivered-order revenue came from **SP, RJ, MG**. Product-category selling-price growth was spread over many categories.
- November new customers in the four selected states: **5,074**; **2,025** purchased in any of the five leading categories (**39.91% reach**), up from **38.53%** in October (+1.38 percentage points).
- November Top5 buyers had **BRL 143.96 AOV** versus **BRL 157.66** among other buyers, an **8.69% lower AOV** for Top5 buyers.

**Business decision:** Do **not** reduce top-five category promotion solely because of AOV. We cannot establish profitability or incremental marketing return without cost of goods, customer acquisition spend, promotional exposure, and longer customer follow-up.

**Scope boundaries:** This is an observational, descriptive analysis. It does **not** include 90-day retention estimates, A/B tests, statistical significance, causality, predictive modeling, or a dashboard. Some orders have missing review/payment/delivery fields; documented reconstruction of one payment is labeled. Category `price` excludes freight, while total payment value includes actual tendered amounts. Dataset ends in 2018, and observed-history customer classification is not lifetime acquisition status.